In [ ]:
!pip install -q transformers accelerate bitsandbytes peft trl \
    qwen-vl-utils pillow datasets wandb

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive_1/",force_remount=True)

from pathlib import Path
import torch
from PIL import Image
from datasets import Dataset

from transformers import (
    Qwen2VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig


### Configure the path ###########
TRAIN_JSON   = "/content/drive_1/MyDrive/data/outputs/drivelm_nusc.json"
IMAGE_ROOT   = "/content/drive_1/MyDrive/data/v1.0-mini"
OUTPUT_DIR   = "/content/drive_1/MyDrive/qwen2vl_qlora"
LOGGING_DIR  = "/content/drive_1/MyDrive/qwen2vl_qlora/logs"
MODEL_ID     = "Qwen/Qwen2-VL-2B-Instruct"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOGGING_DIR, exist_ok=True)

In [ ]:
######### Prepare the dataset for training #####################
QA_CATEGORIES = ["perception", "prediction", "planning"]

with open(TRAIN_JSON) as f:
    entries = json.load(f)

rows = []
for entry in entries:
    image_rel = entry.get("cam_front_path", "")
    image_abs = os.path.join(IMAGE_ROOT, image_rel)

    if not os.path.exists(image_abs):
        continue                         # skip if image missing

    qa_paits = entry.get("qa_paits", {})
    if isinstance(qa_paits, dict):
        for cat in QA_CATEGORIES:
            for qa in qa_paits.get(cat, []):
                q = (qa.get("Q") or "").strip()
                a = (qa.get("A") or "").strip()
                if q and a:
                    rows.append({
                        "image_abs": image_abs,
                        "question":  q,
                        "answer":    a,
                        "category":  cat,
                    })

print(f"Training samples : {len(rows)}")
print(f"Category counts  : "
      f"{sum(1 for r in rows if r['category']=='perception')} perception | "
      f"{sum(1 for r in rows if r['category']=='prediction')} prediction | "
      f"{sum(1 for r in rows if r['category']=='planning')} planning")

In [ ]:
## for training the VLM ######
def format_sample(row):
    """
    Returns a dict with:
      - 'messages': the chat-formatted conversation
      - 'image': PIL Image object
    SFTTrainer will call the processor on these.
    """
    image = Image.open(row["image_abs"]).convert("RGB")

    system_content = (
        "You are an autonomous driving assistant. "
        f"Answer the following {row['category']} question about the front camera image."
    )

    messages = [
        {"role": "system",  "content": system_content},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text": row["question"]},
            ],
        },
        {"role": "assistant", "content": row["answer"]},
    ]
    return {"messages": messages, "image": image}

In [ ]:
def find_assistant_start(input_ids, tokenizer):
    """
    Find the token index where the assistant response starts.
    Qwen2-VL uses <|im_start|>assistant as the delimiter.
    """
    assistant_token = tokenizer.encode("<|im_start|>assistant", add_special_tokens=False)
    ids = input_ids.tolist()
    for i in range(len(ids) - len(assistant_token)):
        if ids[i:i+len(assistant_token)] == assistant_token:
            return i + len(assistant_token) + 1   # +1 for the newline token
    return 0   # fallback: compute loss on everything

def collate_fn(batch):
    formatted = [format_sample(row) for row in batch]

    texts = [
        processor.apply_chat_template(
            item["messages"], tokenize=False, add_generation_prompt=False
        )
        for item in formatted
    ]
    images = [item["image"].resize((448,448)) for item in formatted]


    encoding = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1024,
    )

    labels = encoding["input_ids"].clone()

    # Mask everything before the assistant response
    for i, input_ids in enumerate(encoding["input_ids"]):
        start = find_assistant_start(input_ids, processor.tokenizer)
        labels[i, :start] = -100

    pad_id = processor.tokenizer.pad_token_id
    labels[encoding["input_ids"] == pad_id] = -100

    encoding["labels"] = labels
    return encoding


In [ ]:
## Training Configurations ###
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)


model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",    ## CPU/GPU
    trust_remote_code=True,
    torch_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True,
                                          min_pixels=128*28*28, ## due to memory
                                          max_pixels=256*28*28)


if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

# LoRA configuration used #######
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    init_lora_weights="gaussian", ## tighter initialization
    task_type=TaskType.CAUSAL_LM,

    target_modules=r"^(?!.*visual\.blocks)(?!.*merger).*(?:q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj).*"
)



# Build HuggingFace Dataset
hf_dataset = Dataset.from_list(rows)

split = hf_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]

print(f"Train: {len(train_dataset)}  |  Eval (sanity): {len(eval_dataset)}")



training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,        # effective batch = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    fp16=False,                          # with fp=True and bf=False, there is error in TPU
    bf16=True,
    optim="paged_adamw_8bit",
    logging_dir=LOGGING_DIR,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,                   # keep only the 3 most recent checkpoints
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    disable_tqdm=False,
    dataloader_num_workers=2,
    remove_unused_columns=False,
    report_to="none",
    dataset_kwargs={"skip_prepare_dataset":True}
)


trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collate_fn,
    peft_config=lora_config,
)

print("Starting QLoRA fine-tuning …")
trainer.train()



#   save the adapter inorder to reload cheaply for further experiments, then
#   optionally merge for deployment.

adapter_path = os.path.join(OUTPUT_DIR, "lora_adapter")
model.save_pretrained(adapter_path)
processor.save_pretrained(adapter_path)
print(f"LoRA adapter saved → {adapter_path}")



MERGE_AND_SAVE = True   # set True for deployment checkpoint

if MERGE_AND_SAVE:
    print("Merging adapter into base weights …")
    merged_model = model.merge_and_unload()
    merged_path  = os.path.join(OUTPUT_DIR, "merged_model")
    merged_model.save_pretrained(merged_path, safe_serialization=True)
    processor.save_pretrained(merged_path)
    print(f"Merged model saved → {merged_path}")